# FinalTeamProjectAAI-500

In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [77]:
df = pd.read_csv("diabetic_data.csv")
df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

In [78]:
df_clean = df.copy()

## Data Cleaning/Preparation

Removed 'encounter_id', 'patient_nbr', and 'payer_code' from the dataset, given that our objective for this project is to identify which variables are more strongly related to hospital readmissions.

In [79]:

df_clean = df_clean.drop(columns=[
    'encounter_id', 'patient_nbr','payer_code'])
df_clean.columns

Index(['race', 'gender', 'age', 'weight', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'medical_specialty', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses',
       'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
       'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
       'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

In [80]:

# to see how many rows in the dataset has

num_row = df_clean.shape[0]
print(num_row)

101766


### Checking "?" Values

In [81]:
# to see how many ? are in each column
sum_q = (df_clean == '?').sum()
amount_q =sum_q[sum_q>0]
print(num_row - amount_q)

race                  99493
weight                 3197
medical_specialty     51817
diag_1               101745
diag_2               101408
diag_3               100343
dtype: int64


In [82]:
for col in amount_q.index:
    print(col)
    print("Amount Missing : ", sum_q[col])
    print("Amount Available: ", (num_row - amount_q[col]) )
    print("Perentage Missing:" , round((sum_q[col]/num_row)*100,2))

race
Amount Missing :  2273
Amount Available:  99493
Perentage Missing: 2.23
weight
Amount Missing :  98569
Amount Available:  3197
Perentage Missing: 96.86
medical_specialty
Amount Missing :  49949
Amount Available:  51817
Perentage Missing: 49.08
diag_1
Amount Missing :  21
Amount Available:  101745
Perentage Missing: 0.02
diag_2
Amount Missing :  358
Amount Available:  101408
Perentage Missing: 0.35
diag_3
Amount Missing :  1423
Amount Available:  100343
Perentage Missing: 1.4


Weight, with 96.86% missing values, can be removed from the dataset because it does not provide enough information to determine whether it has a strong relationship with hospital readmission.

In [83]:
df_clean = df_clean.drop(columns=[
    'weight'])

### <h3 style = "color:red;">!!! TEAM DECISION NEEDED </h3>


medical_specialty, with 49.08% missing values, has a lot of missing information, but we also have to look at the description of the dataset, where it says that it is the name of the doctor's specialty. This makes us question if their specialty has anything to do with readmission.

### Checking "Yes" and "No" Values

In [84]:
# check to see which have yes and no 
temp = []
for column in df_clean.columns:
    if 'Yes' in df_clean[column].values or 'No' in df_clean[column].values:
        temp.append(column)
print(temp)

['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [85]:
df_clean[temp].apply(lambda x:x.value_counts())

,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,...,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
Ch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47011.0,NaN
Down,575.0,45.0,11.0,1.0,194.0,NaN,560.0,564.0,NaN,118.0,...,NaN,NaN,12218.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN
No,81778.0,100227.0,101063.0,101680.0,96575.0,101765.0,89080.0,91116.0,101743.0,94438.0,...,101766.0,101766.0,47383.0,101060.0,101753.0,101765.0,101764.0,101765.0,54755.0,23403.0
Steady,18346.0,1384.0,668.0,79.0,4670.0,1.0,11356.0,9274.0,23.0,6976.0,...,NaN,NaN,30849.0,692.0,13.0,1.0,2.0,1.0,NaN,NaN
Up,1067.0,110.0,24.0,6.0,327.0,NaN,770.0,812.0,NaN,234.0,...,NaN,NaN,11316.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN
Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78363.0


Looking at the dataset, examide and citoglipton both have only "No" for all 101,766 observations, showing that these variables aren't good candidates for identifying a relationship with hospital readmission.

In [86]:
df_clean =df_clean.drop(columns=['examide','citoglipton'])

To get a better look at the variables acetohexamide, tolbutamide, glipizide-metformin, glimepiride-pioglitazone, metformin-rosiglitazone, metformin-pioglitazone, change, and diabetesMed, I examined their value counts.

In [87]:
l = ['acetohexamide', 'tolbutamide', 'glipizide-metformin', 'glimepiride-pioglitazone','metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']
df_clean[l].apply(lambda x:x.value_counts())



,acetohexamide,tolbutamide,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
Ch,NaN,NaN,NaN,NaN,NaN,NaN,47011.0,NaN
No,101765.0,101743.0,101753.0,101765.0,101764.0,101765.0,54755.0,23403.0
Steady,1.0,23.0,13.0,1.0,2.0,1.0,NaN,NaN
Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78363.0


Looking at the dataset, acetohexamide, glimepiride-pioglitazone, and metformin-pioglitazone have only one person who is "Steady," meaning the patient took their medicine and the dosage did not change. These variables are strong candidates for removal because they give us almost no variation when investigating the relationship with hospital readmission. For this reason, they will be taken out of the dataset.

In [88]:
print(df_clean.columns.tolist())

['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [89]:
df_clean =df_clean.drop(columns=['acetohexamide', 'glimepiride-pioglitazone', 'metformin-pioglitazone'])


In [90]:
temp = [x for x in temp if x not in['acetohexamide', 'glimepiride-pioglitazone', 'metformin-pioglitazone','examide','citoglipton']]
df_clean[temp].apply(lambda x:x.value_counts())

,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,metformin-rosiglitazone,change,diabetesMed
Ch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47011.0,NaN
Down,575.0,45.0,11.0,1.0,194.0,560.0,564.0,NaN,118.0,87.0,3.0,5.0,NaN,NaN,12218.0,6.0,NaN,NaN,NaN,NaN
No,81778.0,100227.0,101063.0,101680.0,96575.0,89080.0,91116.0,101743.0,94438.0,95401.0,101458.0,101728.0,101763.0,101727.0,47383.0,101060.0,101753.0,101764.0,54755.0,23403.0
Steady,18346.0,1384.0,668.0,79.0,4670.0,11356.0,9274.0,23.0,6976.0,6100.0,295.0,31.0,3.0,38.0,30849.0,692.0,13.0,2.0,NaN,NaN
Up,1067.0,110.0,24.0,6.0,327.0,770.0,812.0,NaN,234.0,178.0,10.0,2.0,NaN,1.0,11316.0,8.0,NaN,NaN,NaN,NaN
Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78363.0


To get a better look at the variables tolbutamide, troglitazone, glipizide-metformin, and metformin-rosiglitazone, I examined their value counts.	

In [91]:
med_list = ['tolbutamide', 'troglitazone', 'glipizide-metformin', 'metformin-rosiglitazone'	]
df_clean[med_list].apply(lambda x:x.value_counts())

,tolbutamide,troglitazone,glipizide-metformin,metformin-rosiglitazone
No,101743,101763,101753,101764
Steady,23,3,13,2


Looking at the data for tolbutamide, troglitazone, glipizide-metformin,and metformin-rosiglitazone,the results are same to the previous variables. "Steady" has few observations compared to "No". This shows that these are good candidates for removal since they have little variation.

In [92]:
df_clean =df_clean.drop(columns=['tolbutamide', 'troglitazone', 'glipizide-metformin', 'metformin-rosiglitazone'])


### Checking NaN Values

In [93]:
n= df_clean.isna().sum()
amount_n = n[n>0].sort_values()
print(amount_n)

A1Cresult        84748
max_glu_serum    96420
dtype: int64


Check how much "None," which is stored as NaN in pandas, takes up of the total variable.

In [94]:
percent = (amount_n / len(df_clean))*100
print(round(percent,2))

A1Cresult        83.28
max_glu_serum    94.75
dtype: float64


Looking at the percentage, NaN values take up almost all of max_glu_serum, as well as most of A1Cresult. In the variable description table, it says that if the value is None, the test was not taken. Since they were not taken, we cannot determine whether they contributed to readmission. For this reason, they will be removed from the dataset.

In [95]:
df_clean =df_clean.drop(columns=['A1Cresult', 'max_glu_serum'])


### Cheking "Unknown/ Invalid"

While looking at some of the data, I noticed "Unknown/Invalid" values. We will check these to see if they contribute to readmission and if not, we will remove them from the dataset.

In [96]:
sum_u = (df_clean == 'Unknown/Invalid').sum()
amount_u =sum_u[sum_u>0]
print(amount_u)

gender    3
dtype: int64


### <h3 style = "color:red;">!!! TEAM DECISION NEEDED </h3>

Only gender has 'Unknown/Invalid' values, and there are only three of them, which isn't enough information to analyze. We can possibly remove those three rows.

### Final Cleaned Dataset

In [97]:
df_clean.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,...,pioglitazone,rosiglitazone,acarbose,miglitol,tolazamide,insulin,glyburide-metformin,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,41,0,...,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),1,1,7,3,?,59,0,...,No,No,No,No,No,Up,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),1,1,7,2,?,11,5,...,No,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),1,1,7,2,?,44,1,...,No,No,No,No,No,Up,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),1,1,7,1,?,51,0,...,No,No,No,No,No,Steady,No,Ch,Yes,NO


# Exploratory Data Analysis

# Model Analysis